# Corrective Filter Evaluation

The corrective filter decides per pull request whether a change corrects an intrinsic defect, fixing the domain on which attribution operates. It is calibrated here as a zero-shot in-context classifier over the manually annotated cohort corpus, reducing the multilabel annotation to the binary intrinsic task. Calibration varies prompt formulation, model, and input representation, and selects the operating configuration under a recall-first rule.


In [1]:
import os, json, pathlib, threading, re
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import polars as pl
import plotly.graph_objects as go
import jinja2
from openai import OpenAI
from tqdm import tqdm

from benchmark_notebook.io import CloudStorage
from benchmark_notebook import theme  # noqa: F401  registers thesis template

store = CloudStorage(bucket="bug-request-bucket")

client = OpenAI(
    api_key=os.environ["TOGETHER_AI_API_KEY"], base_url="https://api.together.xyz/v1"
)

annotation = pl.from_arrow(
    store.asset(
        key="filter-change-annotation",
        version="1.0.0",
        location="main",
        partition="default",
    )
)
corpus = pl.from_arrow(
    store.asset(
        key="filter-change-corpus",
        version="1.0.0",
        location="main",
        partition="default",
    )
)
labels = annotation.select(
    pl.col("repository").alias("name"),
    "number",
    pl.col("categories").alias("labels"),
)

content = corpus.join(labels, on=["name", "number"], how="inner").unique(
    subset=["name", "number"]
)

content

name,number,title,body,files,commits,labels
str,i64,str,str,list[struct[4]],list[str],list[str]
"""microsoft/Olive""",2326,"""Gracefully handle new release …","""Gracefully handle new release …","[{""olive/evaluator/lmeval_ort.py"",""MODIFIED"",6,1}]","[""Gracefully handle new release of lm_eval""]","[""extrinsic""]"
"""microsoft/Olive""",248,"""Refactor engine to support har…","""make engine register the Pass …","[{""docs/source/tutorials/advanced_users.md"",""MODIFIED"",2,5}, {""examples/directml/stable_diffusion/stable_diffusion.py"",""MODIFIED"",2,1}, … {""test/unit_test/engine/test_engine.py"",""MODIFIED"",81,30}]","[""make engine register the Pass config instead of Pass instance to support hardware accelerators"", ""add missed run file"", … ""remove accelerator spec in the cached path for model, run, evaluation since the model id contains the info""]","[""other""]"
"""microsoft/Olive""",968,"""try to fix the CI failure""","""""","[{"".azure_pipelines/olive-ci.yaml"",""MODIFIED"",0,1}, {""examples/stable_diffusion/user_script.py"",""MODIFIED"",1,1}]","[""try to fix the CI failure"", ""include the stable diffusion in CI""]","[""intrinsic"", ""other""]"
"""microsoft/Olive""",1095,"""Fix unused bug in gptq_utils""","""fix unused bug in gptq_utils, …","[{""olive/passes/pytorch/gptq_utils.py"",""MODIFIED"",1,1}]","[""fix bug in gpuq_utils""]","[""intrinsic""]"
"""microsoft/Olive""",2307,"""Update unit tests to specify g…","""With newer NNCF, it is necessa…","[{""olive/passes/openvino/compression.py"",""MODIFIED"",333,107}, {""test/passes/openvino/test_openvino_compression.py"",""MODIFIED"",23,3}]","[""Update unit tests to specify group size fallback With newer NNCF, it is necessary to specify group size fallback in advanced compression parameters, which is reflected in a few unit tests that fail due to InvalidGroupSizeError. Also updated the Intel® NNCF Compression pass in Olive OpenVINO passes to accurately handle advanced parameters in both enum and string forms - [ ] Add unit tests for this change. - [x] Make sure all tests can pass. - [ ] Update documents if necessary. - [x] Lint and apply fixes to your code by running `lintrunner -a` - [ ] Is this a user-facing change? If yes, give a description of this change to be included in the release notes.""]","[""intrinsic"", ""other""]"
…,…,…,…,…,…,…
"""microsoft/Olive""",1561,"""Hf data preprocessing: Truncat…","""For huggingface data preproces…","[{""olive/data/component/pre_process_data.py"",""MODIFIED"",9,4}]","[""select data""]","[""other""]"
"""microsoft/Olive""",2301,"""Add with_prior_preservation op…","""Add with_prior_preservation op…","[{""olive/cli/diffusion_lora.py"",""MODIFIED"",30,1}, {""olive/passes/diffusers/lora.py"",""MODIFIED"",210,42}]","[""Add with_prior_preservation option for dreambooth""]","[""other""]"
"""microsoft/Olive""",2355,"""Fix ONNX save crash when Model…","""Summary This PR fixes a crash …","[{""olive/passes/onnx/common.py"",""MODIFIED"",19,9}]","[""Handle ByteSize encode errors by forcing external data save""]","[""intrinsic""]"


In [2]:
names = content.select(pl.col("name")).unique().to_series().implode()

edges = (
    pl.from_arrow(
        store.asset(
            key="scope-reference-edge",
            version="1.0.0",
            location="main",
            partition="default",
        )
    )
    .filter(
        pl.col("name").is_in(names)
        & pl.col("kind").is_in(["closing", "connected"])
        & (pl.col("name") == pl.col("related").struct.field("repository"))
    )
    .select(
        "name",
        "number",
        pl.col("related").struct.field("number").alias("related_number"),
        pl.col("related").struct.field("kind").alias("related_kind"),
    )
)

issues = (
    pl.from_arrow(
        store.asset(
            key="scope-issue-raw",
            version="6.0.0",
            location="main",
            partition="default",
            columns=["name", "type", "number", "payload"],
        )
    )
    .filter(pl.col("name").is_in(names) & (pl.col("type") == "issue"))
    .select(
        "name",
        pl.col("number").cast(pl.Int64),
        pl.col("payload").str.json_path_match("$.title").fill_null("").alias("title"),
        pl.col("payload").str.json_path_match("$.body").fill_null("").alias("body"),
        pl.lit("issue").alias("related_kind"),
    )
)

linked = (
    edges.join(
        issues,
        left_on=["name", "related_number"],
        right_on=["name", "number"],
        how="inner",
    )
    .unique(subset=["name", "number", "related_number"], keep="first")
    .sort("name", "number", "related_number")
    .group_by("name", "number")
    .agg(
        pl.struct(
            pl.col("related_number").alias("number"),
            pl.col("title"),
            pl.col("body"),
        ).alias("issues")
    )
)

comments = (
    store.scan(key="scope-change-raw", version="6.0.0")
    .filter(pl.col("name").is_in(names))
    .filter(pl.col("type") == "comment")
    .select(
        "name",
        pl.col("number").cast(pl.Int64),
        pl.col("payload")
        .str.json_decode(
            pl.List(
                pl.Struct(
                    {
                        "body": pl.Utf8,
                        "created_at": pl.Utf8,
                        "user": pl.Struct({"login": pl.Utf8, "type": pl.Utf8}),
                    }
                )
            )
        )
        .alias("items"),
    )
    .explode("items")
    .unnest("items")
    .unnest("user")
    .filter(
        (pl.col("type") != "Bot")
        & ~pl.col("login").str.ends_with("[bot]")
        & pl.col("body").is_not_null()
        & (pl.col("body").str.strip_chars() != "")
    )
    .sort("name", "number", "created_at")
    .group_by("name", "number")
    .agg(pl.struct("login", pl.col("body").str.strip_chars()).alias("comments"))
    .collect()
)

rows = (
    content.join(linked, on=["name", "number"], how="left")
    .join(comments, on=["name", "number"], how="left")
    .with_columns(
        pl.col("issues").fill_null([]),
        pl.col("comments").fill_null([]),
    )
    .select("name", "number", "title", "body", "files", "commits", "issues", "comments")
    .to_dicts()
)

# Prompt Generation

Four meaning-preserving prompt variants are generated from the annotation guidelines along the surface axes markup, paraphrase, and elaboration. The constraint reproduces every definition, decision rule, and edge case unchanged, so that variation isolates surface formulation from construct content.

In [ ]:
from pathlib import Path
import itertools, random
from jinja2 import Environment, FileSystemLoader

env = Environment(
    loader=FileSystemLoader(Path("prompts")),
    autoescape=False,
    keep_trailing_newline=True,
)
meta = env.get_template("generator.j2")


def vary(prompt, model: str = "Qwen/Qwen3.7-Max"):
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        stream=True,
    )
    parts = []
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            parts.append(delta)
    return "".join(parts)


axes = {
    "markup": ["plain", "markdown"],
    "paraphrase": ["verbatim", "light", "heavy"],
    "elaboration": ["minimal", "full"],
}

out = Path("prompts/generated")
out.mkdir(parents=True, exist_ok=True)
for i, prompt in enumerate(
    [
        meta.render(
            guidelines=Path("guidelines.md").read_text(encoding="utf-8"),
            variation=variant,
        )
        for variant in random.Random(42).sample(
            [dict(zip(axes, values)) for values in itertools.product(*axes.values())], 4
        )
    ]
):
    template = vary(prompt)
    (out / f"{i:02d}.j2").write_text(template, encoding="utf-8")

## Inference

Each candidate pairs a prompt variant with an input representation, ranging from the baseline of title, description, and changed files to variants that add linked issues. The classifier reads each pull request and returns the intrinsic decision, and predictions are cached per configuration so that the calibration can be re-run without repeating completed work.


In [3]:
def unique_first_lines(messages):
    return list(dict.fromkeys(m.strip().splitlines()[0] for m in messages if m.strip()))


def cache(prompt, features, tag, suffix=""):
    parts = [f for f in ("files", "fewshot", "issues") if f in features]
    feats_name = "-".join(parts) if parts else "base"
    p = pathlib.Path(".experiments") / prompt / (feats_name + suffix)
    p.mkdir(parents=True, exist_ok=True)
    return p / f"{tag}.jsonl"


def render(row, prompt, features):
    template = jinja.from_string(prompts[prompt])
    return template.render(
        title=row["title"],
        body=row["body"],
        files=row.get("files", []),
        commits=row.get("commits", []),
        comments=row.get("comments", []),
        issues=row.get("issues", []),
        features=set(features),
    )


def call(model, content, temperature=0.0):
    stream = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": content}],
        temperature=temperature,
        stream=True,
    )
    parts = []
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            parts.append(delta)
    return "".join(parts)


def work(model, row, prompt, feats, temperature=0.0):
    content = render(row, prompt, feats)
    try:
        out = json.loads(call(model, content, temperature=temperature))
    except Exception as e:
        out = {"error": str(e)}
    return row, out


def infer(rows, prompt, feats, tag, model, temperature=0.0, suffix=""):
    path = cache(prompt, feats, tag, suffix=suffix)
    done = set()
    if path.exists():
        for l in path.open():
            if l.strip():
                d = json.loads(l)
                if "labels" in d:
                    done.add((d["name"], d["number"]))
    todo = [r for r in rows if (r["name"], r["number"]) not in done]
    if not todo:
        return
    lock = threading.Lock()
    with path.open("a") as f, ThreadPoolExecutor(max_workers=16) as pool:
        futures = [
            pool.submit(work, model, r, prompt, feats, temperature) for r in todo
        ]
        for fut in tqdm(
            as_completed(futures),
            total=len(todo),
            desc=f"{tag} {prompt}/{path.parent.name}",
        ):
            r, out = fut.result()
            with lock:
                f.write(
                    json.dumps({"name": r["name"], "number": r["number"], **out}) + "\n"
                )
                f.flush()

In [5]:
models = {
    "deepseek-v4": "deepseek-ai/DeepSeek-V4-Pro",
    # "qwen3.7-max": "Qwen/Qwen3.7-Max",
    "glm-5.1": "zai-org/GLM-5.1",
    "glm-5.2": "zai-org/GLM-5.2",
    "kimi-k2.6": "moonshotai/Kimi-K2.6",
    "minimax-m3": "MiniMaxAI/MiniMax-M3",
    "minimax-m2.7": "MiniMaxAI/MiniMax-M2.7",
}

prompts = {
    p.stem: p.read_text()
    for p in sorted(pathlib.Path("prompts/generated").glob("*.j2"))
}

candidates = [
    (prompt, features)
    for prompt in prompts
    for features in [
        (),
        # ("files",),
        # ("issues",),
        # ("files", "fewshot"),
        # ("files", "issues"),
    ]
]


jinja = jinja2.Environment(
    trim_blocks=True,
    lstrip_blocks=True,
    keep_trailing_newline=False,
    autoescape=False,
    undefined=jinja2.StrictUndefined,
)

jinja.filters["unique_first_lines"] = unique_first_lines

In [5]:
for prompt, features in candidates:
    for tag, model in models.items():
        infer(rows, prompt, features, tag, model)

minimax-m3 03/base: 100%|██████████| 5/5 [00:01<00:00,  4.42it/s]


## Metrics

The binary intrinsic decision is scored as precision, recall, and $\hat{F_1}$ against the audited labels. A single set of bootstrap resamples is shared across all candidates and models, which is the precondition for the paired comparison: differences between configurations are read on the same resamples rather than on independently drawn ones.


In [6]:
def feats_key(feats):
    parts = [f for f in ("files", "fewshot", "issues") if f in feats]
    return "-".join(parts) if parts else "base"


def score(y, p):
    tp = int((y & p).sum())
    fp = int((~y & p).sum())
    fn = int((y & ~p).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1


def boot(p):
    out = np.empty((boot_idx.shape[0], 3))
    for i, idx in enumerate(boot_idx):
        out[i] = score(y_true[idx], p[idx])
    return out


def load_preds(prompt, feats, tag):
    path = cache(prompt, feats, tag)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return None
    preds = pl.read_ndjson(path)
    cols = preds.columns

    # neues Format: {"intrinsic": true/false}
    from_bool = (
        pl.col("intrinsic").cast(pl.Boolean)
        if "intrinsic" in cols
        else pl.lit(None, dtype=pl.Boolean)
    )
    # altes Format: {"labels": ["Intrinsic", ...]}
    from_labels = (
        pl.col("labels").cast(pl.List(pl.Utf8)).list.contains("Intrinsic")
        if "labels" in cols
        else pl.lit(None, dtype=pl.Boolean)
    )

    preds = preds.with_columns(
        pl.coalesce([from_bool, from_labels]).fill_null(False).alias("is_intrinsic")
    )

    joined = content.select("name", "number").join(
        preds.unique(subset=["name", "number"]).select(
            "name", "number", "is_intrinsic"
        ),
        on=["name", "number"],
        how="left",
        maintain_order="left",
    )
    return joined["is_intrinsic"].fill_null(False).to_numpy()


y_true = content["labels"].list.contains("intrinsic").to_numpy()
boot_idx = np.random.default_rng(0).integers(0, len(y_true), size=(10000, len(y_true)))

feats_by_key = {feats_key(f): f for _, f in candidates}

rows = []
for prompt, features in candidates:
    for tag, model in models.items():
        p = load_preds(prompt, features, tag)
        if p is None:
            continue
        prec, rec, f1 = score(y_true, p)
        rows.append(
            dict(
                prompt=prompt,
                feats=feats_key(features),
                tag=tag,
                precision=prec,
                recall=rec,
                f1=f1,
            )
        )
scores = pl.DataFrame(rows)

## Configuration Selection

The $\hat{F_1}$ confidence interval delimits the strong group, since at this sample size it is more stable than that of either component. Within the strong group recall is the primary criterion, because the filter is the only stage at which a true corrective change can be removed before attribution observes it, so omissions are costlier than false positives. The least costly configuration is retained, cost increasing with input richness and model size.

In [7]:
ITO = ["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00", "#CC79A7"]


def build_boots(scores):
    return {
        (r["prompt"], r["feats"], r["tag"]): boot(
            load_preds(r["prompt"], feats_by_key[r["feats"]], r["tag"])
        )
        for r in scores.iter_rows(named=True)
    }


def boot_summary(scores, boots, how="mean"):
    rows = []
    for (feats, tag), g in scores.group_by(["feats", "tag"], maintain_order=True):
        keys = [
            (p, feats, tag) for p in g["prompt"].to_list() if (p, feats, tag) in boots
        ]
        if not keys:
            continue

        # F1: Bootstrap-CI über die Prompt-Mittelung
        f1s = np.vstack([boots[k][:, 2] for k in keys])
        agg = f1s.mean(0) if how == "mean" else f1s.max(0)
        f1, f1_lo, f1_hi = float(agg.mean()), *np.percentile(agg, [2.5, 97.5])

        # P, R: Spanne über die Prompt-Punktschätzer
        rows.append(
            dict(
                prompt=how,
                feats=feats,
                tag=tag,
                precision=float(g["precision"].mean()),
                prec_lo=float(g["precision"].min()),
                prec_hi=float(g["precision"].max()),
                recall=float(g["recall"].mean()),
                recall_lo=float(g["recall"].min()),
                recall_hi=float(g["recall"].max()),
                f1=f1,
                ci_lo=f1_lo,
                ci_hi=f1_hi,
            )
        )
    return pl.DataFrame(rows)


def forest(df, row_px=40):
    df = df.sort("f1", descending=True).with_columns(pl.col("tag").alias("label"))
    order = df["label"].to_list()
    n = df.height
    fig = go.Figure()
    for i, tag in enumerate(df.get_column("tag").unique(maintain_order=True)):
        d = df.filter(pl.col("tag") == tag)
        fig.add_trace(
            go.Scatter(
                x=d["f1"].to_list(),
                y=d["label"].to_list(),
                mode="markers",
                name=tag,
                marker=dict(size=6, color=ITO[i % len(ITO)]),
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=(d["ci_hi"] - d["f1"]).to_list(),
                    arrayminus=(d["f1"] - d["ci_lo"]).to_list(),
                    thickness=1.0,
                    width=3,
                ),
                customdata=list(
                    zip(
                        d["precision"].to_list(),
                        d["recall"].to_list(),
                        d["ci_lo"].to_list(),
                        d["ci_hi"].to_list(),
                    )
                ),
                hovertemplate=(
                    "<b>%{y}</b><br>F1 %{x:.3f}  "
                    "[%{customdata[2]:.3f}, %{customdata[3]:.3f}]<br>"
                    "P %{customdata[0]:.3f}   R %{customdata[1]:.3f}<extra></extra>"
                ),
            )
        )
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=list(reversed(order)),
        range=[-0.6, n - 0.4],
    )
    fig.update_layout(
        xaxis_title="F1",
        xaxis_range=[0.5, 1.0],
        template="simple_white",
        height=80 + row_px * n,
        showlegend=False,
        width=600,
        margin=dict(l=10, r=10, t=46, b=38),
    )
    return fig


base = scores.filter(pl.col("feats") == "base")
boots = build_boots(base)
forest(boot_summary(base, boots)).show(
    config={"toImageButtonOptions": {"format": "svg", "filename": "plot", "scale": 1}}
)

In [8]:
def fmt(col):
    return col.map_elements(lambda x: f"{x:.2f}", return_dtype=pl.Utf8)


def span(value, lo, hi):
    return fmt(pl.col(value)) + " [" + fmt(pl.col(lo)) + ", " + fmt(pl.col(hi)) + "]"


def show_table(df):
    df = df.sort("f1", descending=True).select(
        pl.col("tag").alias("Model"),
        span("precision", "prec_lo", "prec_hi").alias("Precision (range)"),
        span("recall", "recall_lo", "recall_hi").alias("Recall (range)"),
        span("f1", "ci_lo", "ci_hi").alias("F1 (95% CI)"),
    )
    with pl.Config(
        tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
    ):
        print(df)


def show_feats(scores, boots, model, prompt="00"):
    rows = []
    sub = scores.filter((pl.col("tag") == model) & (pl.col("prompt") == prompt))
    for r in sub.iter_rows(named=True):
        key = (r["prompt"], r["feats"], r["tag"])
        if key not in boots:
            continue
        f1s = boots[key][:, 2]
        lo, hi = np.percentile(f1s, [2.5, 97.5])
        rows.append(
            dict(
                feats=r["feats"],
                precision=r["precision"],
                recall=r["recall"],
                f1=float(f1s.mean()),
                ci_lo=float(lo),
                ci_hi=float(hi),
            )
        )
    df = (
        pl.DataFrame(rows)
        .sort("f1", descending=True)
        .select(
            pl.col("feats").alias("Features"),
            fmt(pl.col("precision")).alias("Precision"),
            fmt(pl.col("recall")).alias("Recall"),
            span("f1", "ci_lo", "ci_hi").alias("F1 (95% CI)"),
        )
    )
    with pl.Config(
        tbl_rows=-1, tbl_hide_dataframe_shape=True, tbl_hide_column_data_types=True
    ):
        print(df)


base = scores.filter(pl.col("feats") == "base")
boots = build_boots(base)
show_table(boot_summary(base, boots))

model = "deepseek-v4"
sub = scores.filter(pl.col("tag") == model)
boots = build_boots(sub)
show_feats(sub, boots, model, prompt="00")

┌──────────────┬───────────────────┬───────────────────┬───────────────────┐
│ Model        ┆ Precision (range) ┆ Recall (range)    ┆ F1 (95% CI)       │
╞══════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│ deepseek-v4  ┆ 0.84 [0.80, 0.89] ┆ 0.76 [0.70, 0.80] ┆ 0.79 [0.71, 0.86] │
│ glm-5.2      ┆ 0.87 [0.79, 0.95] ┆ 0.70 [0.67, 0.76] ┆ 0.77 [0.68, 0.85] │
│ kimi-k2.6    ┆ 0.89 [0.85, 0.93] ┆ 0.67 [0.65, 0.69] ┆ 0.76 [0.67, 0.84] │
│ glm-5.1      ┆ 0.91 [0.88, 0.95] ┆ 0.65 [0.61, 0.67] ┆ 0.75 [0.67, 0.83] │
│ minimax-m3   ┆ 0.96 [0.94, 0.97] ┆ 0.60 [0.57, 0.65] ┆ 0.74 [0.63, 0.83] │
│ minimax-m2.7 ┆ 0.83 [0.79, 0.84] ┆ 0.63 [0.57, 0.70] ┆ 0.71 [0.61, 0.80] │
└──────────────┴───────────────────┴───────────────────┴───────────────────┘
┌──────────┬───────────┬────────┬───────────────────┐
│ Features ┆ Precision ┆ Recall ┆ F1 (95% CI)       │
╞══════════╪═══════════╪════════╪═══════════════════╡
│ base     ┆ 0.89      ┆ 0.78   ┆ 0.83 [0.74, 0.90] │
└──────────┴──

## Error Analysis

The retained configuration is inspected through its residual false negatives and false positives. The error rate is broken down by label mix and by label count, which exposes where the residual errors concentrate, in particular among changes that combine an intrinsic correction with other work. A stubborn-case view ranks the misclassified pull requests by how many configurations across the full grid also failed them, separating configuration-specific errors from cases that resist every configuration.

In [9]:
best = scores.sort("f1", descending=True).row(0, named=True)
preds_best = load_preds(best["prompt"], feats_by_key[best["feats"]], best["tag"])
fn_mask = (y_true == 1) & (preds_best == 0)
fp_mask = (y_true == 0) & (preds_best == 1)
print(f"best: {best['prompt']} / {best['feats']} / {best['tag']}  F1={best['f1']:.3f}")
print(f"FN={int(fn_mask.sum())}  FP={int(fp_mask.sum())}")

best: 00 / base / deepseek-v4  F1=0.832
FN=12  FP=5


In [10]:
def pr_table(mask):
    return (
        content.filter(pl.Series(mask))
        .with_columns(
            (
                "https://github.com/"
                + pl.col("name")
                + "/pull/"
                + pl.col("number").cast(pl.Utf8)
            ).alias("url")
        )
        .select("url", "title", "labels")
    )


print("=== False Negatives ===")
print(pr_table(fn_mask))
print("\n=== False Positives ===")
print(pr_table(fp_mask))

=== False Negatives ===
shape: (12, 3)
┌─────────────────────────────────┬─────────────────────────────────┬────────────────────────┐
│ url                             ┆ title                           ┆ labels                 │
│ ---                             ┆ ---                             ┆ ---                    │
│ str                             ┆ str                             ┆ list[str]              │
╞═════════════════════════════════╪═════════════════════════════════╪════════════════════════╡
│ https://github.com/microsoft/O… ┆ Update unit tests to specify g… ┆ ["intrinsic", "other"] │
│ https://github.com/microsoft/O… ┆ 📝 Update wiki/example use      ┆ ["intrinsic", "other"] │
│                                 ┆ case…                           ┆                        │
│ https://github.com/microsoft/O… ┆ Model Splitting: Add cost mode… ┆ ["intrinsic", "other"] │
│ https://github.com/awslabs/gra… ┆ Change per etype fanout to can… ┆ ["intrinsic"]          │
│ https://gi

In [11]:
def error_breakdown(errors):
    print("by label mix")
    for label in ("intrinsic", "extrinsic", "other"):
        mask = content["labels"].list.contains(label).to_numpy()
        if mask.sum() == 0:
            continue
        print(
            f"  contains {label!r}: {int(errors[mask].sum())}/{int(mask.sum())} = {errors[mask].mean():.2%}"
        )
    print("by label count")
    nl = content["labels"].list.len().to_numpy()
    for k in sorted(set(nl)):
        mask = nl == k
        print(
            f"  {k} labels: {int(errors[mask].sum())}/{int(mask.sum())} = {errors[mask].mean():.2%}"
        )


errors_best = preds_best != y_true
error_breakdown(errors_best)

by label mix
  contains 'intrinsic': 12/54 = 22.22%
  contains 'extrinsic': 0/13 = 0.00%
  contains 'other': 10/158 = 6.33%
by label count
  1 labels: 12/174 = 6.90%
  2 labels: 5/24 = 20.83%
  3 labels: 0/1 = 0.00%


## Majority Vote

The retained production configuration is deepseek-v4 on the baseline prompt at greedy decoding. A single greedy pass cannot express its own uncertainty, so the same configuration is resampled at nonzero temperature and the per pull request decision is taken as the majority over an odd number of independent runs. Self-consistency turns the sampling variance into a vote, which trades a small amount of the greedy pass's determinism for robustness on borderline changes. The vote is scored as $\hat{F_1}$ with a bootstrap interval on the shared resamples, so it is directly comparable to the single-pass rows above.

In [12]:
VOTE_PROMPT, VOTE_FEATS, VOTE_TAG = "00", (), "deepseek-v4"
VOTE_TEMP = 0.7
VOTE_SUFFIXES = ["-0", "-1", "-2"]  # ungerade Anzahl Voter → keine Ties

# `rows` ist ab der Metrik-Zelle mit den Score-Zeilen belegt; die PR-Dicts für
# die Inferenz werden hier neu aufgebaut (identisch zur Korpus-Zelle oben).
pr_rows = (
    content.join(linked, on=["name", "number"], how="left")
    .join(comments, on=["name", "number"], how="left")
    .with_columns(
        pl.col("issues").fill_null([]),
        pl.col("comments").fill_null([]),
    )
    .select("name", "number", "title", "body", "files", "commits", "issues", "comments")
    .to_dicts()
)

for suffix in VOTE_SUFFIXES:
    infer(
        pr_rows,
        VOTE_PROMPT,
        VOTE_FEATS,
        VOTE_TAG,
        models[VOTE_TAG],
        temperature=VOTE_TEMP,
        suffix=suffix,
    )


def load_run(prompt, feats, tag, suffix):
    # wie load_preds, aber mit suffix an cache() durchgereicht
    path = cache(prompt, feats, tag, suffix=suffix)
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return None
    preds = pl.read_ndjson(path)
    cols = preds.columns
    from_bool = (
        pl.col("intrinsic").cast(pl.Boolean)
        if "intrinsic" in cols
        else pl.lit(None, dtype=pl.Boolean)
    )
    from_labels = (
        pl.col("labels").cast(pl.List(pl.Utf8)).list.contains("Intrinsic")
        if "labels" in cols
        else pl.lit(None, dtype=pl.Boolean)
    )
    decision = pl.coalesce([from_bool, from_labels])
    # Ein PR, der erst einen Fehler warf und beim Resume erfolgreich nachgeholt
    # wurde, steht doppelt in der Datei (Fehlerzeile + gültige Zeile). Gültige
    # Vorhersagen werden vor der Deduplizierung bevorzugt, damit die Fehlerzeile
    # die echte nicht überschreibt.
    preds = (
        preds.with_columns(
            decision.alias("decision"),
            decision.is_not_null().alias("has_pred"),
        )
        .sort("has_pred", descending=True)
        .unique(subset=["name", "number"], keep="first")
    )
    joined = content.select("name", "number").join(
        preds.select("name", "number", "decision"),
        on=["name", "number"],
        how="left",
        maintain_order="left",
    )
    return joined["decision"].fill_null(False).to_numpy()


votes = np.vstack(
    [load_run(VOTE_PROMPT, VOTE_FEATS, VOTE_TAG, s) for s in VOTE_SUFFIXES]
)
majority = votes.sum(0) >= (len(VOTE_SUFFIXES) // 2 + 1)

prec, rec, f1 = score(y_true, majority)
lo, hi = np.percentile(boot(majority)[:, 2], [2.5, 97.5])
print(
    f"majority({len(VOTE_SUFFIXES)}× {VOTE_TAG} @T={VOTE_TEMP}, {VOTE_PROMPT}/base)  "
    f"P={prec:.2f} R={rec:.2f} F1={f1:.2f} [{lo:.2f}, {hi:.2f}]"
)


majority(3× deepseek-v4 @T=0.7, 00/base)  P=0.86 R=0.78 F1=0.82 [0.73, 0.89]
